# SSH spectral clustering — summer

Thin driver notebook: loads summer SSH spectra, runs PCA + GMM clustering,
and reproduces the thesis figures for this season using the
`ssh_clustering` package (see `src/ssh_clustering/`).

Run `pip install -e ..` (or `pip install -e "..[notebooks]"`) from the
`notebooks/` directory first if the package isn't installed yet.


In [ ]:
import matplotlib.pyplot as plt

from ssh_clustering import config, masks, plotting
from ssh_clustering.pipeline import run_season_pipeline

DATA_ROOT = "../data/Global"   # see ../data/README.md for expected layout
MASK_PATH = "../data/mask_to_plot_ratio.npz"

SEASON = "summer"


## Run the pipeline

This loads and preprocesses the spectra, fits PCA, scans GMM cluster counts
via BIC/AIC/silhouette score, and fits the final model with
`config.FINAL_N_CLUSTERS` components (currently 4 — see `config.py` to
change it).

In [ ]:
result = run_season_pipeline(season=SEASON, data_root=DATA_ROOT)

print(f"{len(result.dataset.file_paths)} locations loaded")
print(f"PCA reduced to {result.spectra_pca.shape[1]} components")


## Model selection: choosing the number of clusters

In [ ]:
ms = result.model_selection
print(f"Optimal n (BIC): {ms.optimal_n_bic}")
print(f"Optimal n (AIC): {ms.optimal_n_aic}")
print(f"Optimal n (Silhouette): {ms.optimal_n_silhouette}")

plotting.plot_model_selection(ms, title_suffix=f"({SEASON})")
plt.show()


## Final clustering

In [ ]:
plotting.plot_pca_scatter(
    result.spectra_pca, result.clustering_result.labels, title=f"GMM clusters ({SEASON})"
)
plt.show()

print("Cluster sizes:")
print(result.cluster_df["labels"].value_counts().sort_index())


Percentages (fraction of locations per cluster — not area-weighted; map area and this percentage can diverge since the map below is built by nearest-neighbor interpolation onto an unweighted lat/lon grid).

In [ ]:
from ssh_clustering import diagnostics

percentages = diagnostics.cluster_size_percentages(result.clustering_result.labels)
print(percentages)


## Per-cluster diagnostics

Mean spectrum and RMS spread of member spectra around that mean, per
cluster — a cohesion check, not part of the clustering itself.

In [ ]:
for diag in result.cluster_diagnostics:
    plotting.plot_mean_spectrum(diag, title=f"{SEASON} — cluster {diag.cluster_label}")
    plt.show()
    print(f"Cluster {diag.cluster_label}: mean RMS = {diag.mean_rms:.2f}")


## Global cluster distribution map

Uses the descriptive regime names configured in
`config.SEASON_CLUSTER_NAMES["summer"]`. Update those (and `config.FINAL_N_CLUSTERS`)
together if you change the number of clusters.

In [ ]:
land_mask = masks.load_land_mask(MASK_PATH)
cluster_names = config.SEASON_CLUSTER_NAMES[SEASON]

plotting.plot_cluster_map(
    lon=result.dataset.lon,
    lat=result.dataset.lat,
    labels=result.clustering_result.labels,
    mask=land_mask,
    cluster_names=cluster_names,
    dx=config.GRID_DX,
    lon_range=config.GRID_LON_RANGE,
    lat_range=config.GRID_LAT_RANGE,
    title=f"Global cluster distribution — {SEASON}",
    regime_colors=config.REGIME_COLORS,
)
plt.show()
